In [3]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/drive/MyDrive/FightAttention /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
!cp -r /content/FightAttention/run/ViolenceDetection.py .

In [11]:
!cp -r /content/FightAttention/run/utilities.py .

In [16]:
!cp -r /content/FightAttention/violence_yolo.onnx .
!cp -r /content/FightAttention/temporal_classifier.onnx .
!cp -r /content/FightAttention/temporal_classifier.onnx.data .

In [8]:
!pip install onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 82.6 MB/s eta 0:00:00


In [12]:
from ViolenceDetection import ViolenceDetector
import os


In [13]:
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, roc_curve, auc, roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def compute_metrics(y_true, y_pred, y_probs=None, threshold=0.5):
    """
    Compute comprehensive classification metrics

    Args:
        y_true: Ground truth labels (0/1)
        y_pred: Predicted labels (0/1)
        y_probs: Predicted probabilities (optional, for ROC/AUC)
        threshold: Classification threshold
    """

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # Basic Metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    # Additional Metrics
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0  # False Negative Rate
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # False Positive Rate

    # ROC-AUC (if probabilities provided)
    roc_auc = None
    if y_probs is not None:
        roc_auc = roc_auc_score(y_true, y_probs)

    # Print Results
    print(f"\n{'='*50}")
    print(f"CONFUSION MATRIX (Threshold: {threshold})")
    print(f"{'='*50}")
    print(f"TP: {tp:4d} | FP: {fp:4d}")
    print(f"FN: {fn:4d} | TN: {tn:4d}")
    print(f"\n{'='*50}")
    print(f"METRICS")
    print(f"{'='*50}")
    print(f"Accuracy:    {accuracy:.4f}")
    print(f"Precision:   {precision:.4f}")
    print(f"Recall:      {recall:.4f}")
    print(f"F1 Score:    {f1:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"FPR:         {fpr:.4f}")
    print(f"FNR:         {fnr:.4f}")
    if roc_auc is not None:
        print(f"ROC-AUC:     {roc_auc:.4f}")

    # Classification Report
    print(f"\n{'='*50}")
    print(f"CLASSIFICATION REPORT")
    print(f"{'='*50}")
    print(classification_report(y_true, y_pred, target_names=['No Violence', 'Violence']))

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'specificity': specificity,
        'fpr': fpr,
        'fnr': fnr,
        'roc_auc': roc_auc,
        'cm': cm
    }

def plot_confusion_matrix(y_true, y_pred, save_path='confusion_matrix.png'):
    """Plot and save confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Violence', 'Violence'],
                yticklabels=['No Violence', 'Violence'])
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    logger.info(f"Confusion matrix saved to {save_path}")
    plt.close()

def plot_roc_curve(y_true, y_probs, save_path='roc_curve.png'):
    """Plot and save ROC curve"""
    fpr, tpr, thresholds = roc_curve(y_true, y_probs)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    logger.info(f"ROC curve saved to {save_path}")
    plt.close()

def threshold_analysis(y_true, y_probs, save_path='threshold_analysis.png'):
    """Analyze metrics across different thresholds"""
    thresholds = np.arange(0, 1.01, 0.05)
    metrics_by_threshold = {
        'precision': [],
        'recall': [],
        'f1': [],
        'accuracy': []
    }

    for thresh in thresholds:
        y_pred = (y_probs >= thresh).astype(int)
        metrics_by_threshold['precision'].append(precision_score(y_true, y_pred, zero_division=0))
        metrics_by_threshold['recall'].append(recall_score(y_true, y_pred, zero_division=0))
        metrics_by_threshold['f1'].append(f1_score(y_true, y_pred, zero_division=0))
        metrics_by_threshold['accuracy'].append(accuracy_score(y_true, y_pred))

    plt.figure(figsize=(10, 6))
    for metric, values in metrics_by_threshold.items():
        plt.plot(thresholds, values, marker='o', label=metric)
    plt.xlabel('Classification Threshold')
    plt.ylabel('Metric Value')
    plt.title('Metrics vs Classification Threshold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    logger.info(f"Threshold analysis saved to {save_path}")
    plt.close()

def analyze(y_pred, y_true, y_scores, dataset_name=""):

    y_pred = (np.array(y_scores) >= 0.5).astype(int)

    # Compute metrics
    metrics = compute_metrics(y_true, y_pred, y_scores, threshold=0.5)

    # Generate visualizations
    plot_confusion_matrix(y_true, y_pred, save_path=f'{dataset_name}_confusion_matrix.png')
    plot_roc_curve(y_true, y_scores, save_path=f'{dataset_name}_roc_curve.png')
    threshold_analysis(y_true, y_scores, save_path=f'{dataset_name}_threshold_analysis.png')

    return metrics

In [ ]:
VAL_DIRECTORY = "RWF_val"  # Update this path to your validation directory
TP, FP, TN, FN = 0, 0, 0, 0

y_true = []
y_scores = []

#Fight validation
for video_file in os.listdir(VAL_DIRECTORY + "/Fight"):
    video_path = os.path.join(VAL_DIRECTORY, "fight", video_file)
    detector = ViolenceDetector(video_path)
    max_violence_prob, _ = detector.val()
    y_true.append(1)  # Assuming this is a fight video
    y_scores.append(max_violence_prob)
    if max_violence_prob >= 0.5:  # Threshold for classification
        TP += 1
    else:
        FN += 1

#Non-fight validation
for video_file in os.listdir(VAL_DIRECTORY + "/NonFight"):
    video_path = os.path.join(VAL_DIRECTORY, "NonFight", video_file)
    detector = ViolenceDetector(video_path)
    max_violence_prob, _ = detector.val()
    y_true.append(0)  # Assuming this is a non-fight video
    y_scores.append(max_violence_prob)
    if max_violence_prob < 0.5:  # Threshold for classification
        TN += 1
    else:
        FP += 1

metrics = analyze(y_true, y_true, y_scores, VAL_DIRECTORY)


In [ ]:
VAL_DIRECTORY = "hkfval"

y_true = []
y_scores = []

for video_file in os.listdir(VAL_DIRECTORY):
    if not video_file.endswith(".avi"):
        continue

    fight = video_file[:2] == "fi"
    video_path = os.path.join(VAL_DIRECTORY, video_file)

    detector = ViolenceDetector(video_path)
    max_violence_prob, _ = detector.val()

    # store results
    y_true.append(1 if fight else 0)
    y_scores.append(max_violence_prob)

metrics = analyze(y_true, y_true, y_scores, VAL_DIRECTORY)

In [ ]:
VAL_DIRECTORY = "Peliculas"  # Update this path to your validation directory
TP, FP, TN, FN = 0, 0, 0, 0

y_true = []
y_scores = []

#Fight validation
for video_file in os.listdir(VAL_DIRECTORY + "/fights"):
    video_path = os.path.join(VAL_DIRECTORY, "fights", video_file)
    detector = ViolenceDetector(video_path)
    max_violence_prob, _ = detector.val()
    y_true.append(1)  # Assuming this is a fight video
    y_scores.append(max_violence_prob)
    if max_violence_prob >= 0.5:  # Threshold for classification
        TP += 1
    else:
        FN += 1

#Non-fight validation
for video_file in os.listdir(VAL_DIRECTORY + "/nofights"):
    video_path = os.path.join(VAL_DIRECTORY, "nofights", video_file)
    detector = ViolenceDetector(video_path)
    max_violence_prob, _ = detector.val()
    y_true.append(0)  # Assuming this is a non-fight video
    y_scores.append(max_violence_prob)
    if max_violence_prob < 0.5:  # Threshold for classification
        TN += 1
    else:
        FP += 1

metrics = analyze(y_true, y_true, y_scores, VAL_DIRECTORY)


In [18]:
VAL_DIRECTORY = "/content/FightAttention/RLVS"  # Update this path to your validation directory

y_true = []
y_scores = []

#Fight validation
for video_file in os.listdir(VAL_DIRECTORY + "/Violence"):
    video_path = os.path.join(VAL_DIRECTORY, "Violence", video_file)
    detector = ViolenceDetector(video_path)
    max_violence_prob, _ = detector.val()
    y_true.append(1)  # Assuming this is a fight video
    y_scores.append(max_violence_prob)

#Non-fight validation
for video_file in os.listdir(VAL_DIRECTORY + "/NonViolence"):
    video_path = os.path.join(VAL_DIRECTORY, "NonViolence", video_file)
    detector = ViolenceDetector(video_path)
    max_violence_prob, _ = detector.val()
    y_true.append(0)  # Assuming this is a non-fight video
    y_scores.append(max_violence_prob)

metrics = analyze(y_true, y_true, y_scores, VAL_DIRECTORY)


KeyboardInterrupt: 